*0.3 Classical NLP*

# FastText

**The situation.** The Word2Vec model from item 16 was trained on your tickets. Customers write "refunnd", "chargeback", "unsubscribing" — none in the vocabulary, all unknown, all silently dropped from every feature. Typos are 3% of tokens in real tickets.

**FastText.** Word2Vec plus subwords: each word's vector is the sum of vectors for its character n-grams ("<ref", "refu", "efun", …). A misspelled or never-seen word shares most of its n-grams with known words, so it gets a sensible vector instead of nothing. Also trained with gensim.

In [1]:
# Load OPENAI_API_KEY from the .env file. The OpenAI clients read it from the environment.
from dotenv import find_dotenv, load_dotenv

load_dotenv(find_dotenv())
MODEL = "gpt-4o-mini"

In [2]:
import warnings

from gensim.models import FastText
from gensim.utils import simple_preprocess

warnings.filterwarnings("ignore")
from sklearn.datasets import fetch_20newsgroups

topics = ["rec.autos", "rec.motorcycles", "sci.med", "sci.space", "comp.graphics"]
posts = fetch_20newsgroups(
    subset="train", categories=topics, remove=("headers", "footers", "quotes"), random_state=0
)
sentences = []
for post in posts.data:
    tokens = simple_preprocess(post)
    if len(tokens) > 5:
        sentences.append(tokens)

model = FastText(
    sentences,
    vector_size=100,
    window=5,
    min_count=5,
    workers=4,
    epochs=10,
    seed=0,
    min_n=3,
    max_n=5,
)
print("vocabulary:", len(model.wv), "words")
for unseen in ("orbitting", "motorcyle", "spacecrafts"):
    print(
        
            f"{unseen!r} in vocabulary: {unseen in model.wv.key_to_index}  → nearest known: "
            f"{model.wv.most_similar(unseen, topn=3)[0][0]}"
        
    )
assert (
    "motorcyle" not in model.wv.key_to_index
    and model.wv.similarity("motorcyle", "motorcycle") > 0.8
)

Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'


vocabulary: 7792 words
'orbitting' in vocabulary: False  → nearest known: orbiting
'motorcyle' in vocabulary: False  → nearest known: motorcycle
'spacecrafts' in vocabulary: False  → nearest known: spacecraft


**Reading the output.** None of the three misspelled or unseen words is in the vocabulary, yet each gets a vector close to the word it resembles — built from shared character pieces. Word2Vec would have raised `KeyError` for all three. (It is not magic: a typo that shares few pieces with the right word, like "docter", can land somewhere odd.)

```
"motorcyle"   → <mo, mot, oto, tor, orc, rcy, cyl, yle, le>      character 3-grams
"motorcycle"  → <mo, mot, oto, tor, orc, rcy, cyc, ycl, cle, le>  most pieces shared
                 vector(motorcyle) ≈ vector(motorcycle)
```

**The rule to remember.** FastText gives every string a vector, seen or not, through its character pieces. Use it where typos and rare words are common — tickets, chat, product names.

| Use it when | Don't when | Instead use |
|---|---|---|
| noisy user text; morphologically rich languages (German, Finnish, Turkish) | clean text with a stable vocabulary — Word2Vec is smaller and faster | Word2Vec |

**Watch out**
- Model files are large (every n-gram has a vector); `min_n`/`max_n` control size.
- A vector for nonsense ("qzxv") is still produced; check similarity before trusting it.
- Still one vector per word, no context; for sentence meaning use sentence embeddings.